# M4-FIT — turning bench data into a corrected model

**BruSafe / CAPSID · iGEM 2026 · Module 4 — Assembly**

This notebook is self-contained: the first code cell writes the simulation source to disk, so nothing has to be downloaded. It runs on Colab, in Jupyter, or on a laptop with `numpy`, `scipy` and `matplotlib`.

The theory and the bench protocols (§11) are described in [`M4_theory_compendium.pdf`](../../docs/M4_theory_compendium.pdf) in `capsid/docs/`. The data template, `M4_bench_data_template.csv` (in `capsid/data/`), is read from `../../data/`; if it contains no values, the notebook falls back to a synthetic example (§1).

The other notebooks of the series make predictions. This one allows a prediction to be wrong and then updates it. The workflow is:

1. fill in `M4_bench_data_template.csv` at the bench,
2. run this notebook,
3. it returns the fitted $g_{contact}$, $k_a$ and a recovery scale, with uncertainties,
4. and it plots the prior curve and the posterior curve on the same axes.

**What is fitted and what is not.** Three parameters are fitted: the per-contact free energy $g_{contact}$ (determined by *where* the optimum lies), the association rate $k_a$ (determined by *how high* the peak is and how long the lag lasts), and a single recovery scale $s_{rec}\in(0,1]$ that absorbs purification and gel-sampling losses. Without the third parameter every handling loss would be attributed to the physics, and $g_{contact}$ would come out systematically too weak.

The critical nucleus $n_c$ is **fixed at 4** and not fitted. From a concentration series alone, $n_c$ and $g_{contact}$ are confounded: a larger nucleus and a weaker contact give almost the same curve. Fitting both would give a tight-looking result that is an artefact. $g_{contact}$ should therefore be reported as *conditional on* $n_c=4$.

In [ ]:
# Colab / fresh machine: uncomment
# !pip -q install numpy scipy matplotlib
import numpy as np, matplotlib.pyplot as plt
np.set_printoptions(precision=4, suppress=True)

In [ ]:
%%writefile brusafe_synthesis.py
"""
BruSafe - MS2 VLP synthesis: expanded mathematical model
=========================================================
Two modules, self-contained (numpy + scipy only):

  Module 3+  Coat-protein SYNTHESIS: transcription / translation ODEs with an
             EXPLICIT reversible monomer<->dimer equilibrium.  The proposal's
             assumption D = P/2 is recovered as the tight-binding limit (Ni 1995).

  Module 4+  Capsid SELF-ASSEMBLY as an EXPLICIT sequential nucleation-elongation
             cascade over all 90 intermediate sizes (Zlotnick 1994; Endres &
             Zlotnick 2002; Ceres & Zlotnick 2002; Hagan 2010).  Reverse rates
             come from subunit-subunit contact free energies via detailed
             balance, so the model reproduces the nucleation lag and the
             concentration-dependent KINETIC TRAP that the lumped 3-state ODE
             in the main proposal cannot show.

All parameters carry literature-backed values; every rate is estimable without
experimental data (Mode A) and refinable against EMSA/SEC/light-scattering time
courses if available (Mode B).
"""
import numpy as np
from scipy.integrate import solve_ivp, odeint

RT    = 0.593    # kcal/mol at ~298 K
N_CAP = 90       # coat-protein dimers per T=3 MS2 capsid (Ni 1995)

# =====================================================================
# MODULE 3+  Expression with explicit reversible dimerisation
# =====================================================================
def induction(t, t_on=30.0, ramp=10.0):
    """Smooth inducer switch (IPTG / arabinose): 0 -> 1 around t_on."""
    return 1.0 / (1.0 + np.exp(-(t - t_on) / ramp))

def expression_ode(y, t, alpha, delta_m, mu, k_tl, delta_P, t_on):
    """dm/dt = alpha*u(t) - (delta_m+mu) m ;  dP/dt = 1e-3 k_tl m - (delta_P+mu) P
    m in nM, P in uM monomer-equivalents (1e-3 converts nM mRNA -> uM protein)."""
    m, P = y
    u = induction(t, t_on)
    dm = alpha * u - (delta_m + mu) * m
    dP = 1e-3 * k_tl * m - (delta_P + mu) * P
    return [dm, dP]

def simulate_expression(p, t_grid, t_on=30.0):
    delta_m = np.log(2) / p["t_half_mrna_min"]
    mu      = np.log(2) / p["t_double_min"]
    sol = odeint(expression_ode, [0.0, 0.0], t_grid,
                 args=(p["alpha_tx_nM_min"], delta_m, mu,
                       p["k_tl_permin"], p["delta_prot_permin"], t_on),
                 rtol=1e-8, atol=1e-10)
    m, P = sol[:, 0], sol[:, 1]
    return m, P

def expression_steady_state(p):
    """Closed form (u->1): m_ss = alpha/(delta_m+mu);
       P_ss = 1e-3 k_tl m_ss / (delta_P+mu).  Pen-and-paper estimable."""
    delta_m = np.log(2) / p["t_half_mrna_min"]
    mu      = np.log(2) / p["t_double_min"]
    m_ss = p["alpha_tx_nM_min"] / (delta_m + mu)
    P_ss = 1e-3 * p["k_tl_permin"] * m_ss / (p["delta_prot_permin"] + mu)
    return m_ss, P_ss

def monomer_dimer_split(P_tot_uM, Kd_dim_uM):
    """Equilibrium partition of total protein into free monomer M and dimer D:
         2D + M = P_tot ,  D = M^2 / Kd_dim.
    Solve (2/Kd)M^2 + M - P_tot = 0.  As Kd_dim -> 0 (tight dimer, Ni 1995),
    D -> P_tot/2, which is the assumption used in the main proposal."""
    a = 2.0 / Kd_dim_uM; b = 1.0; c = -P_tot_uM
    M = (-b + np.sqrt(b*b - 4*a*c)) / (2*a)
    D = M*M / Kd_dim_uM
    return M, D

# =====================================================================
# MODULE 4+  Explicit sequential nucleation-elongation cascade
#   x[j-1] = concentration (uM) of the j-mer, j = 1..N_CAP
#   x_1 is the free coat-protein DIMER (the assembly subunit)
#   reaction j:  X_j + X_1 <-> X_{j+1}   (j=1 uses x_1^2)
# =====================================================================
def contacts_for_step(j_target, n_c):
    """Subunit-subunit contacts formed when a subunit adds to make the j_target-mer.
    Pre-nucleus (<= n_c): 1 contact -> net UP-hill with the entropic penalty ->
      nucleation is rare (the free-energy barrier).
    Growth: 2 contacts.  Closing vertices (last ~8): 3 contacts -> strong drive
      to completion (Endres & Zlotnick 2002)."""
    if j_target <= n_c:            return 1
    if j_target >= N_CAP - 8:      return 3
    return 2

def build_rates(k_a, g_assoc, g_contact, n_c):
    """Forward on-rate k_a (uM^-1 min^-1); reverse from detailed balance.
      dG_add(j) = g_assoc + contacts(j)*g_contact
      g_assoc  > 0 : per-subunit association (translational/rotational) penalty
      g_contact< 0 : free energy per subunit-subunit contact (Ceres 2002)
      Kd(j) = exp(dG_add/RT) ;  b_{j+1} = k_a * Kd(j)."""
    f = np.full(N_CAP, k_a)
    b = np.zeros(N_CAP + 1)
    for j in range(1, N_CAP):
        c = contacts_for_step(j + 1, n_c)
        dG_add = g_assoc + c * g_contact
        b[j + 1] = k_a * np.exp(dG_add / RT)
    return f, b

def cascade_ode(t, x, f, b):
    x = np.clip(x, 0.0, None)
    x1 = x[0]
    J = np.zeros(N_CAP - 1)                # J[j-1] = net flux of reaction j
    J[0] = f[0] * x1 * x1 - b[2] * x[1]    # 2 X1 -> X2
    for j in range(2, N_CAP):
        J[j - 1] = f[j - 1] * x[j - 1] * x1 - b[j + 1] * x[j]
    dx = np.zeros(N_CAP)
    dx[0] = -2 * J[0] - np.sum(J[1:])      # subunit consumed by every step
    dx[1] = J[0] - (J[1] if N_CAP > 2 else 0.0)
    for k in range(3, N_CAP):
        dx[k - 1] = J[k - 2] - J[k - 1]
    dx[N_CAP - 1] = J[N_CAP - 2]           # complete capsid
    return dx

def simulate_cascade(D0_uM, k_a, g_assoc, g_contact, n_c, t_max=240.0, nt=600):
    """Returns t, capsid_frac(t), free_frac(t), inter_frac(t), full state X.
    Mass fractions are of the total subunit pool and sum to 1 at all times."""
    f, b = build_rates(k_a, g_assoc, g_contact, n_c)
    x0 = np.zeros(N_CAP); x0[0] = D0_uM
    t_eval = np.linspace(0, t_max, nt)
    sol = solve_ivp(cascade_ode, (0, t_max), x0, t_eval=t_eval,
                    args=(f, b), method="LSODA", rtol=1e-7, atol=1e-10)
    x = sol.y
    capsid_frac = N_CAP * x[-1] / D0_uM
    free_frac   = x[0] / D0_uM
    j = np.arange(1, N_CAP + 1)[:, None]
    inter_frac  = (np.sum(j * x, axis=0) - x[0] - N_CAP * x[-1]) / D0_uM
    return t_eval, capsid_frac, free_frac, inter_frac, x

# =====================================================================
# Literature-mean parameters (Mode A defaults)
# =====================================================================
PARAMS_SYNTH = {
    # Module 3 expression (Milo 2010; Bernstein 2002; Selinger 2003)
    "t_half_mrna_min":  (5.0,  (2.0, 8.0),  "Bernstein2002; Selinger2003"),
    "t_double_min":     (40.0, (25., 60.),  "Milo2010"),
    "alpha_tx_nM_min":  (6.0,  (1.0, 30.),  "Milo2010"),
    "k_tl_permin":      (4.0,  (1.0, 12.),  "Milo2010"),
    "delta_prot_permin":(0.002,(0.0, 0.01), "Milo2010"),
    "Kd_dim_uM":        (0.01, (1e-3, 1.0), "Ni1995 (tight obligate dimer)"),
    # Module 4 assembly (Zlotnick1994; Endres2002; Ceres2002; Hagan2010)
    "k_a_uM_min":       (3.0,  (0.4, 4.0),  "Ceres2002; Endres2002 (assoc on-rate)"),
    "g_assoc_kcal":     (6.0,  (4.0, 8.0),  "Zlotnick1994 (assoc entropy penalty)"),
    "g_contact_kcal":   (-4.0, (-5., -3.),  "Ceres2002 (per-contact free energy)"),
    "n_c_nucleus":      (4,    (3, 6),      "Endres2002 (nucleus size)"),
}

if __name__ == "__main__":
    p = {k: v[0] for k, v in PARAMS_SYNTH.items()}
    m_ss, P_ss = expression_steady_state(p)
    M, D = monomer_dimer_split(P_ss, p["Kd_dim_uM"])
    print(f"M3+ steady state: mRNA={m_ss:.1f} nM, protein={P_ss:.2f} uM, "
          f"dimer={D:.2f} uM ({2*D/P_ss:.3f} of mass; P/2={P_ss/2:.2f})")
    for D0 in (2.0, 10.0):
        t, cf, ff, itf, _ = simulate_cascade(
            D0, p["k_a_uM_min"], p["g_assoc_kcal"], p["g_contact_kcal"], p["n_c_nucleus"])
        print(f"M4+ D0={D0:5} uM: capsid@240={cf[-1]:.2f}  intermediate@240={itf[-1]:.2f}"
              f"  ({'trapped' if itf[-1] > cf[-1] else 'productive'})")


In [ ]:
%%writefile M4_fit.py
"""
M4_fit.py -- fit the Module 4 assembly model to benchtop data.
BruSafe / CAPSID, iGEM 2026.

Floated parameters
    g_contact   per-contact free energy (kcal/mol)  -- set by the OPTIMUM POSITION
    k_a         association rate (uM^-1 min^-1)     -- set by the PEAK HEIGHT and the LAG
    s_rec       one multiplicative recovery scale in (0,1], absorbing purification and
                gel/TEM sampling losses. Without it, handling loss is mis-attributed
                to the physics.

Fixed on purpose
    n_c = 4       confounded with g_contact from concentration data alone; fix it and
                  report g_contact as CONDITIONAL on it (compendium, section 7)
    g_assoc = +6.0
    N = 90        structural, not fitted

Requires brusafe_synthesis.py in the same directory.
"""
import numpy as np
from scipy.optimize import least_squares
import brusafe_synthesis as L1

N_CAP = 90
J = np.arange(1, N_CAP + 1)


def run_one(D0, k_a, g_contact, n_c=4, g_assoc=6.0, t_max=240.0):
    """Thin wrapper: returns (t, capsid_frac, free_frac, inter_frac, x) from the L1 engine."""
    return L1.simulate_cascade(D0_uM=float(D0), k_a=float(k_a), g_assoc=float(g_assoc),
                               g_contact=float(g_contact), n_c=int(n_c), t_max=float(t_max))


def predict_yield(D0, k_a, g_contact, n_c=4, g_assoc=6.0, t_max=240.0):
    """Closed-capsid mass fraction at t_max."""
    _, cap, _, _, _ = run_one(D0, k_a, g_contact, n_c, g_assoc, t_max)
    return float(cap[-1])


def predict_curve(D0s, k_a, g_contact, n_c=4, g_assoc=6.0, t_max=240.0):
    return np.array([predict_yield(d, k_a, g_contact, n_c, g_assoc, t_max) for d in D0s])


def predict_turbidity(D0s, k_a, g_contact, n_c=4, g_assoc=6.0, t_max=240.0):
    """Rayleigh turbidity ~ sum_j x_j j^2, normalised so a fully assembled sample reads 1.
    Partial shells scatter too -- which is exactly why turbidity alone cannot prove
    assembly, and why it must be read jointly with a size-resolving assay."""
    out = []
    for d in D0s:
        _, _, _, _, x = run_one(d, k_a, g_contact, n_c, g_assoc, t_max)
        out.append(float((x[:, -1] * J**2).sum() / (float(d) * N_CAP)))
    return np.array(out)


def fit_yield_curve(D_meas, Y_meas, sigma=None, n_c=4, g_assoc=6.0, t_max=240.0,
                    x0=(3.0, -4.0, 0.8), bounds=((0.2, -5.5, 0.2), (8.0, -2.5, 1.0))):
    """Joint fit of a whole concentration series. Returns the estimates, an approximate
    standard error from the Jacobian, the residuals and the RMSE."""
    D_meas = np.asarray(D_meas, float)
    Y_meas = np.asarray(Y_meas, float)
    w = np.ones_like(Y_meas) if sigma is None else 1.0 / np.asarray(sigma, float)

    def resid(p):
        k_a, gc, s = p
        return (s * predict_curve(D_meas, k_a, gc, n_c, g_assoc, t_max) - Y_meas) * w

    s = least_squares(resid, list(x0), bounds=bounds, diff_step=0.06,
                      xtol=1e-4, ftol=1e-4, max_nfev=200)
    dof = max(len(Y_meas) - len(s.x), 1)
    s2 = float(2 * s.cost / dof)
    try:
        sd = np.sqrt(np.clip(np.diag(np.linalg.inv(s.jac.T @ s.jac) * s2), 0, None))
    except np.linalg.LinAlgError:
        sd = np.full(3, np.nan)
    return dict(k_a=float(s.x[0]), g_contact=float(s.x[1]), s_rec=float(s.x[2]),
                sd=sd, resid=np.asarray(s.fun), rmse=float(np.sqrt(np.mean(s.fun**2))),
                success=bool(s.success), n_c=int(n_c), g_assoc=float(g_assoc))


def profile_g_contact(D_meas, Y_meas, grid, n_c=4, g_assoc=6.0, t_max=240.0):
    """Profile likelihood in g_contact: at each fixed value re-optimise k_a and s_rec.
    Gives a valid interval when parameters are correlated. Returns (grid, rmse)."""
    D_meas = np.asarray(D_meas, float); Y_meas = np.asarray(Y_meas, float)
    out = []
    for gc in grid:
        def resid(p):
            return p[1] * predict_curve(D_meas, p[0], gc, n_c, g_assoc, t_max) - Y_meas
        s = least_squares(resid, [3.0, 0.8], bounds=([0.2, 0.2], [8.0, 1.0]),
                          diff_step=0.06, xtol=1e-4, ftol=1e-4, max_nfev=80)
        out.append(float(np.sqrt(np.mean(s.fun**2))))
    return np.asarray(grid, float), np.asarray(out)


def profile_interval(grid, rmse, threshold):
    """Where the profile-RMSE curve crosses `threshold`, by linear interpolation
    between grid points. Reporting bare grid membership instead of the crossing
    collapses the interval to a single point whenever the profile is sharp -- which
    it is here. Returns (lo, hi); either may be NaN if the curve never rises above
    the threshold on that side, meaning the grid is too narrow."""
    grid = np.asarray(grid, float); rmse = np.asarray(rmse, float)
    i = int(np.argmin(rmse))

    def cross(idx_range):
        for a, b in idx_range:
            if (rmse[a] - threshold) * (rmse[b] - threshold) <= 0 and rmse[b] != rmse[a]:
                w = (threshold - rmse[a]) / (rmse[b] - rmse[a])
                return float(grid[a] + w * (grid[b] - grid[a]))
        return float("nan")

    lo = cross([(k, k - 1) for k in range(i, 0, -1)])
    hi = cross([(k, k + 1) for k in range(i, len(grid) - 1)])
    return lo, hi


def optimum_of(k_a, g_contact, n_c=4, g_assoc=6.0, grid=None, t_max=240.0):
    """Position and height of the yield optimum. Third element flags a grid-edge
    optimum, i.e. the peak is outside the scanned window and is not identified."""
    grid = np.geomspace(0.2, 12, 26) if grid is None else np.asarray(grid, float)
    y = predict_curve(grid, k_a, g_contact, n_c, g_assoc, t_max)
    i = int(np.argmax(y))
    return float(grid[i]), float(y[i]), bool(i in (0, len(grid) - 1))


## 1. Loading the data

`DATA` points to the filled-in CSV file. Until then the cell falls back to a synthetic example generated from a known truth, so that the notebook runs end to end before any measurement exists and the recovery of the truth by the fitter can be checked.

In [ ]:
import pandas as pd, numpy as np, M4_fit as F

DATA  = '../../data/M4_bench_data_template.csv'   # path to the filled-in CSV
ASSAY = 'nativePAGE'                   # which assay column to fit

try:
    df = pd.read_csv(DATA)
    df = df[(df.assay == ASSAY) & df.value.notna()]
    assert len(df) >= 4, 'need at least 4 concentrations to identify 3 parameters'
except (FileNotFoundError, AssertionError) as e:
    print('using SYNTHETIC example data --', e)
    D = np.array([0.5, 1.0, 2.0, 3.0, 5.0])
    rng = np.random.default_rng(7)
    truth = dict(k_a=2.2, g_contact=-4.25, s_rec=0.72)
    Yt = truth['s_rec'] * F.predict_curve(D, truth['k_a'], truth['g_contact'])
    df = pd.DataFrame(dict(assay=ASSAY, D0_uM=D, t_min=240,
                           value=np.clip(Yt + rng.normal(0, 0.04, D.size), 0, 1),
                           value_sd=0.05))

D_meas = df.D0_uM.to_numpy(float)
Y_meas = df.value.to_numpy(float)
S_meas = df.value_sd.to_numpy(float) if 'value_sd' in df else None
print(df[['D0_uM', 'value']].to_string(index=False))

## 2. Fit

Bounded least squares. The reported standard errors come from the Jacobian at the optimum and are only as reliable as the linearisation; section 4 gives the profile-likelihood interval for the parameter that matters.

In [ ]:
res = F.fit_yield_curve(D_meas, Y_meas, sigma=S_meas, n_c=4)
print(f"g_contact = {res['g_contact']:+.2f} +/- {res['sd'][1]:.2f} kcal/mol   "
      f"(conditional on n_c = {res['n_c']})")
print(f"k_a       = {res['k_a']:.2f} +/- {res['sd'][0]:.2f} uM^-1 min^-1")
print(f"s_rec     = {res['s_rec']:.2f} +/- {res['sd'][2]:.2f}  (recovery, not physics)")
print(f"RMSE      = {res['rmse']:.3f}   converged = {res['success']}")

## 3. Prior versus posterior

The literature-prior curve, the measured points and the refitted curve are shown on one axis, which makes visible how far the data moved the model.

In [ ]:
Dg = np.geomspace(0.2, 12, 40)
prior = F.predict_curve(Dg, k_a=3.0, g_contact=-4.0)            # literature prior
post  = res['s_rec'] * F.predict_curve(Dg, res['k_a'], res['g_contact'])

fig, ax = plt.subplots(figsize=(6.2, 4.2))
ax.plot(Dg, prior*100, '--', lw=1.8, color='#8aa8a0',
        label='prior (literature parameters)')
ax.plot(Dg, post*100,  '-',  lw=2.4, color='#0b6e5a',
        label='posterior (fitted to your data)')
ax.errorbar(D_meas, Y_meas*100, yerr=(S_meas*100 if S_meas is not None else None),
            fmt='o', ms=7, color='#b3402f', capsize=3, zorder=5, label='measured')
d_pr, y_pr, _ = F.optimum_of(3.0, -4.0)
d_po, y_po, edge = F.optimum_of(res['k_a'], res['g_contact'])
ax.axvline(d_pr, color='#8aa8a0', ls=':', lw=1)
ax.axvline(d_po, color='#0b6e5a', ls=':', lw=1)
ax.set_xscale('log'); ax.set_xlabel(r'initial dimer concentration $[D]_0$ ($\mu$M)')
ax.set_ylabel('closed-capsid yield at 240 min (%)')
ax.set_title('Module 4 assembly optimum: before and after bench data')
ax.legend(frameon=False, fontsize=9)
fig.tight_layout(); fig.savefig('M4_FIT_prior_vs_posterior.png', dpi=300)
print(f'optimum moved {d_pr:.2f} -> {d_po:.2f} uM; peak {y_pr*100:.0f}% -> {y_po*100:.0f}%')
if edge: print('WARNING: fitted optimum sits at the edge of the scanned window - '
               'extend the concentration series, the peak is not identified')

## 4. Profile likelihood on $g_{contact}$: the interval to report

$k_a$ and $g_{contact}$ are correlated, so the Jacobian standard error of section 2 is optimistic. Here $g_{contact}$ is fixed on a grid and the other two parameters are re-optimised at each point. The interval to report is bounded by the points at which the RMSE curve rises above the measurement noise.

In [ ]:
# two-stage grid: coarse sweep, then refine around the minimum. The profile is
# sharp, so a coarse grid alone cannot resolve where it crosses the noise level.
coarse = np.linspace(-5.2, -3.0, 12)
gc0, rc0 = F.profile_g_contact(D_meas, Y_meas, coarse)
c = gc0[int(np.argmin(rc0))]
fine = np.linspace(c - 0.45, c + 0.45, 19)
gc1, rc1 = F.profile_g_contact(D_meas, Y_meas, fine)

g = np.concatenate([gc0, gc1]); rmse = np.concatenate([rc0, rc1])
o = np.argsort(g); g, rmse = g[o], rmse[o]

noise = float(np.nanmean(S_meas)) if S_meas is not None else 0.05
thr = rmse.min() + noise
lo, hi = F.profile_interval(g, rmse, thr)

fig, ax = plt.subplots(figsize=(5.8, 3.9))
ax.plot(g, rmse, 'o-', lw=2, ms=4, color='#0b6e5a')
ax.axhline(thr, color='#b3402f', ls='--', lw=1.2, label=f'min + noise ({noise:.03f})')
if np.isfinite(lo) and np.isfinite(hi):
    ax.axvspan(lo, hi, color='#0b6e5a', alpha=0.12)
ax.set_xlabel(r'$g_{contact}$ (kcal/mol)'); ax.set_ylabel('RMSE of the fit')
ax.set_title(r'Profile likelihood ($n_c$ fixed at 4)')
ax.legend(frameon=False, fontsize=9)
fig.tight_layout(); fig.savefig('M4_FIT_profile.png', dpi=300)

if np.isfinite(lo) and np.isfinite(hi):
    print(f'g_contact = {g[int(np.argmin(rmse))]:+.2f} '
          f'[{lo:+.2f}, {hi:+.2f}] kcal/mol   (n_c = 4)')
    print(f'width = {hi-lo:.2f} kcal/mol   '
          f"(compare the Jacobian SE above: {res['sd'][1]:.2f})")
else:
    print('profile never crosses the noise level inside the grid - widen it, or '
          'your data do not constrain g_contact at all')

## 5. What the correction predicts next

A fit that only reproduces the data it was fitted to has little value; the refitted model should also make new predictions that can be checked. This cell states two.

In [ ]:
d_new = float(np.round(d_po, 2))
y_new = F.predict_yield(d_new, res['k_a'], res['g_contact']) * res['s_rec']
d_trap = float(np.round(d_po * 3, 2))
y_trap = F.predict_yield(d_trap, res['k_a'], res['g_contact']) * res['s_rec']
print('PREDICTION 1 -- run the reaction at the refitted optimum')
print(f'   [D]0 = {d_new} uM  ->  expected yield {y_new*100:.0f}%')
print('PREDICTION 2 -- kinetic trapping above the optimum')
print(f'   [D]0 = {d_trap} uM  ->  expected yield {y_trap*100:.0f}% '
      '(partial shells dominate; TEM should show broken/incomplete particles)')
print()
print('If prediction 2 fails -- high concentration gives GOOD yield -- the '
      'nucleation-elongation picture is wrong for this system, not just mis-parameterised.')

---
**Outputs:** `M4_FIT_prior_vs_posterior.png`, `M4_FIT_profile.png`.

**Reporting checklist.** $g_{contact}$ should be quoted with the *profile* interval and not with the Jacobian one, and stated as conditional on $n_c=4$. $s_{rec}$ should be reported separately and never folded into the physics. The prior curve should be shown next to the posterior so that the change is visible.